# 04 — Knowledge Tracing (BKT)

**Purpose:** Implement Bayesian Knowledge Tracing (BKT) to model student
cognitive mastery over individual knowledge concepts (nodes).

## Theory: Bayesian Knowledge Tracing

BKT is a Hidden Markov Model (HMM) with **4 parameters** per skill:

| Parameter | Symbol | Description |
|-----------|--------|-------------|
| Initial knowledge | $P(L_0)$ | Probability student already knows the skill at start |
| Learning rate | $P(T)$ | Probability of learning on any given practice opportunity |
| Guess rate | $P(G)$ | Probability of correct answer despite not knowing |
| Slip rate | $P(S)$ | Probability of incorrect answer despite knowing |

### Knowledge Update Rule

After observing a correct response:
$$P(L_t | \text{correct}) = \frac{P(L_{t-1})(1 - P(S))}{P(L_{t-1})(1 - P(S)) + (1 - P(L_{t-1})) P(G)}$$

After observing an incorrect response:
$$P(L_t | \text{incorrect}) = \frac{P(L_{t-1}) P(S)}{P(L_{t-1}) P(S) + (1 - P(L_{t-1}))(1 - P(G))}$$

Then apply learning transition:
$$P(L_{t+1}) = P(L_t | \text{evidence}) + (1 - P(L_t | \text{evidence})) \cdot P(T)$$

In [ ]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('..'))

GOLD_DIR = os.environ.get('BDC_GOLD_DIR', '../data/lakehouse/gold')
DUCKDB_PATH = os.environ.get('BDC_DUCKDB_PATH', '../data/student_analytics.duckdb')
%matplotlib inline

print('Setup complete.')

## 1. Load Interaction Data

Try DuckDB first (unified_interactions view), then fall back to
`gold_user_item_matrix` Parquet, then synthetic demo data.

In [ ]:
df_interactions = None

# Try DuckDB
if os.path.exists(DUCKDB_PATH):
    try:
        from scripts.load_data import load_duckdb_view
        df_interactions = load_duckdb_view(
            "SELECT user_id, node_id, action_type, is_correct, created_at "
            "FROM unified_interactions ORDER BY user_id, created_at"
        )
        print(f'[DuckDB] Loaded unified_interactions: {df_interactions.shape}')
    except Exception as e:
        print(f'[DuckDB] Failed: {e}')

# Try Parquet
if df_interactions is None:
    parquet_path = os.path.join(GOLD_DIR, 'gold_user_item_matrix.parquet')
    if os.path.exists(parquet_path):
        df_raw = pd.read_parquet(parquet_path)
        # Simulate is_correct column from struggle data
        rng = np.random.default_rng(0)
        df_interactions = df_raw.copy()
        if 'action_type' not in df_interactions.columns:
            df_interactions['action_type'] = 'quick_check'
        if 'is_correct' not in df_interactions.columns:
            df_interactions['is_correct'] = rng.integers(0, 2, len(df_interactions))
        if 'created_at' not in df_interactions.columns:
            df_interactions['created_at'] = pd.date_range('2024-01-01', periods=len(df_interactions), freq='30min')
        print(f'[Parquet] Loaded gold_user_item_matrix: {df_interactions.shape}')

# Synthetic demo data
if df_interactions is None:
    print('[Demo] Generating synthetic interaction data...')
    rng = np.random.default_rng(42)
    n = 5000
    df_interactions = pd.DataFrame({
        'user_id':    rng.integers(1, 51, n),
        'node_id':    rng.integers(100, 130, n),
        'action_type': rng.choice(['quick_check', 'view', 'learn', 'review'], n,
                                  p=[0.5, 0.25, 0.15, 0.10]),
        'is_correct': rng.integers(0, 2, n),
        'created_at': pd.date_range('2024-01-01', periods=n, freq='15min'),
    })

print(f'Interaction data shape: {df_interactions.shape}')
display(df_interactions.head(5))

## 2. Prepare BKT Dataset

Filter to `quick_check` events (assessment events) and encode `node_id` as
`skill_name` for the BKT model API.

In [ ]:
# Filter to quick_check events
df_bkt = df_interactions[df_interactions['action_type'] == 'quick_check'].copy()
print(f'quick_check events: {len(df_bkt):,} (from {len(df_interactions):,} total interactions)')

if len(df_bkt) == 0:
    print('No quick_check events found; using all interactions as BKT input.')
    df_bkt = df_interactions.copy()

# Encode node_id as skill_name string (pyBKT API expects string skill names)
df_bkt['skill_name'] = 'node_' + df_bkt['node_id'].astype(str)

# Ensure correct column for pyBKT: 'correct' (int 0/1)
if 'is_correct' in df_bkt.columns:
    df_bkt['correct'] = df_bkt['is_correct'].astype(int)
else:
    df_bkt['correct'] = (df_bkt.get('implicit_affinity_score', pd.Series(dtype=float)) > 2.0).astype(int)

# Sort by user and time
if 'created_at' in df_bkt.columns:
    df_bkt = df_bkt.sort_values(['user_id', 'created_at'])
else:
    df_bkt = df_bkt.sort_values('user_id')

bkt_input = df_bkt[['user_id', 'skill_name', 'correct']].rename(columns={'user_id': 'user_id'})

print(f'\nBKT input shape: {bkt_input.shape}')
print(f'Unique skills (nodes): {bkt_input["skill_name"].nunique()}')
print(f'Unique students:       {bkt_input["user_id"].nunique()}')
print(f'Overall accuracy:      {bkt_input["correct"].mean():.3f}')
display(bkt_input.head(10))

## 3. Fit BKT Model

Using `pyBKT` if available. Falls back to a hand-coded BKT implementation.

In [ ]:
mastery_scores = []  # will be populated below

try:
    from pyBKT.models import Model as BKTModel
    print('[pyBKT] Library found. Fitting BKT model...')

    model = BKTModel(seed=42, num_fits=1)
    model.fit(
        data=bkt_input,
        skills='skill_name',
        user_id='user_id',
        correct='correct',
    )

    print('\nFitted model parameters:')
    display(model.params())

    # Predict mastery probability per (user, skill)
    preds = model.predict(
        data=bkt_input,
        skills='skill_name',
        user_id='user_id',
        correct='correct',
    )
    mastery_scores = preds[['user_id', 'skill_name', 'state_predictions']].rename(
        columns={'state_predictions': 'mastery_prob'}
    )
    print(f'\nMastery predictions: {len(mastery_scores):,} rows')

except ImportError:
    print('[SKIP] pyBKT not installed. Using hand-coded BKT implementation...')
    print('  Install with: pip install pyBKT')

    # --- Hand-coded BKT ---
    DEFAULT_PARAMS = {'p_l0': 0.25, 'p_t': 0.10, 'p_g': 0.20, 'p_s': 0.10}

    def bkt_update(p_l, is_correct: int, params: dict) -> float:
        """Update knowledge estimate P(L) after one observation."""
        p_l0 = params['p_l0']
        p_t  = params['p_t']
        p_g  = params['p_g']
        p_s  = params['p_s']
        if is_correct:
            num   = p_l * (1 - p_s)
            denom = p_l * (1 - p_s) + (1 - p_l) * p_g
        else:
            num   = p_l * p_s
            denom = p_l * p_s + (1 - p_l) * (1 - p_g)
        p_l_given_obs = num / denom if denom > 0 else p_l
        return p_l_given_obs + (1 - p_l_given_obs) * p_t

    rows = []
    for (user_id, skill_name), group in bkt_input.groupby(['user_id', 'skill_name']):
        p_l = DEFAULT_PARAMS['p_l0']
        for correct in group['correct']:
            p_l = bkt_update(p_l, correct, DEFAULT_PARAMS)
        rows.append({'user_id': user_id, 'skill_name': skill_name, 'mastery_prob': p_l})

    mastery_scores = pd.DataFrame(rows)
    print(f'Hand-coded BKT mastery scores: {len(mastery_scores):,} (user, skill) pairs')
    print(f'\nDefault BKT parameters used: {DEFAULT_PARAMS}')

if not isinstance(mastery_scores, pd.DataFrame):
    mastery_scores = pd.DataFrame(mastery_scores)

display(mastery_scores.head(10))

## 4. Visualize Mastery Distribution

In [ ]:
if len(mastery_scores) > 0 and 'mastery_prob' in mastery_scores.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Histogram of mastery probabilities
    ax1 = axes[0]
    ax1.hist(mastery_scores['mastery_prob'], bins=30, color='steelblue',
             edgecolor='white', alpha=0.85)
    ax1.axvline(0.8, color='red', linestyle='--', label='Mastery threshold (0.80)')
    ax1.set_title('Distribution of P(Mastery) Scores', fontsize=13, fontweight='bold')
    ax1.set_xlabel('Mastery Probability')
    ax1.set_ylabel('Count')
    ax1.legend()
    ax1.spines[['top', 'right']].set_visible(False)
    ax1.grid(axis='y', alpha=0.3, linestyle='--')

    # Top-10 hardest concepts (lowest avg mastery)
    ax2 = axes[1]
    hardest = (
        mastery_scores.groupby('skill_name')['mastery_prob']
        .mean()
        .sort_values(ascending=True)
        .head(10)
    )
    colors = plt.cm.RdYlGn(hardest.values)
    bars = ax2.barh(hardest.index, hardest.values, color=colors, edgecolor='white')
    ax2.axvline(0.5, color='orange', linestyle='--', alpha=0.8, label='50% mastery')
    ax2.set_title('Top-10 Hardest Knowledge Concepts\n(Lowest Avg Mastery)', fontsize=13, fontweight='bold')
    ax2.set_xlabel('Average Mastery Probability')
    ax2.set_ylabel('Knowledge Node')
    ax2.legend()
    ax2.spines[['top', 'right']].set_visible(False)
    ax2.grid(axis='x', alpha=0.3, linestyle='--')

    plt.tight_layout()
    plt.savefig('../output/bkt_mastery_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Chart saved to ../output/bkt_mastery_distribution.png')

    print(f'\nTop 10 hardest concepts (lowest avg mastery):')
    display(hardest.reset_index().rename(columns={'mastery_prob': 'avg_mastery_prob'}))

    # Stats
    mastered = (mastery_scores['mastery_prob'] >= 0.80).mean()
    print(f'\nFraction of (user, skill) pairs with mastery >= 0.80: {mastered:.1%}')
else:
    print('No mastery scores to visualize.')